Author: Danny Lillard

Date: 2026-04-03

Desc: Testing return of algorithm we will look at the following:

- Buy and hold
- Naive algo
- Clustering algo

In [2]:
# buy and hold
import pandas as pd

raw_data_df = pd.read_csv("quarterly_clusters_model/aa_daily_data.csv")
raw_data_df['timestamp'] = pd.to_datetime(raw_data_df['timestamp'])

start_date = pd.to_datetime("2022-01-03")
end_date = pd.to_datetime("2022-03-31")

start_and_stop_df = raw_data_df[raw_data_df['timestamp'].isin([start_date, end_date])]

# Get the unique tickers in the dataset
tickers = start_and_stop_df['ticker'].unique()

# Calculate the return for each ticker
returns = []
for ticker in tickers:
    ticker_data = start_and_stop_df[start_and_stop_df['ticker'] == ticker]
    ticker_data = ticker_data.sort_values('timestamp')
    
    # Get the opening price on the start date and the closing price on the end date
    start_price = ticker_data[ticker_data['timestamp'] == start_date]['open'].values[0]
    end_price = ticker_data[ticker_data['timestamp'] == pd.to_datetime("2022-03-31")]['close'].values[0]
    
    # Calculate the return
    return_pct = (end_price - start_price) / start_price
    returns.append((ticker,return_pct))

# save returns to csv
returns_df = pd.DataFrame(returns, columns=['ticker', 'return'])
returns_df.to_csv("buy_and_hold_returns.csv", index=False)

# calculate overall return
overall_return = sum([r[1] for r in returns]) / len(returns)
print(f"Overall Buy and Hold Return: {overall_return:.2%}")

Overall Buy and Hold Return: -6.41%


In [ ]:
# seeing returns for naive algo
naive_returns_df = pd.read_csv("signals_and_returns.csv")

naive_returns_df.columns = ['timestamp', 'ticker', 'signal', 'return']

np.float64(0.00035040632087914725)

In [4]:
# zipping returns for clustering algo

clustering_signals = pd.read_csv('quarterly_clusters_model/signals_and_returns.csv', header=None)

clustering_signals.columns = ['timestamp', 'ticker', 'signal']

clustering_signals['timestamp'] = pd.to_datetime(clustering_signals['timestamp'])

raw_data_df = pd.read_csv("quarterly_clusters_model/aa_daily_data.csv")
raw_data_df['timestamp'] = pd.to_datetime(raw_data_df['timestamp'])

clustering_signals_and_returns = pd.merge(clustering_signals, raw_data_df, on=['timestamp', 'ticker'], how='left')

def compute_trade_return(signal: str, open_t: float, close_t: float) -> float:
    if signal == "BUY":
        return (close_t - open_t) / open_t
    elif signal == "SELL":
        return (open_t - close_t) / open_t
    return 0.0

clustering_signals_and_returns['return'] = clustering_signals_and_returns.apply(lambda row: compute_trade_return(row['signal'], row['open'], row['close']), axis=1)


In [7]:
clustering_signals_and_returns['return'].groupby(clustering_signals_and_returns['ticker']).mean().sum()

np.float64(0.034976310231534385)

In [10]:
overall_naive_return = naive_returns_df['return'].groupby(naive_returns_df['ticker']).mean().sum()
overall_naive_return

np.float64(0.07498382268135512)